In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from skimage import util, filters, color
from skimage.segmentation import watershed
from skimage.measure import label

from scipy.stats import norm 
from scipy.spatial.distance import pdist
from scipy.interpolate import make_interp_spline

from stm_voronoi_mst import img_file, gray_process, voronoi_tree, find_nodes, statistics

In [ ]:
def plot_voronoi(img, img_vor, img_vor_deg, v_border, G_inner, g_msf = None,
                 show_borders = True, show_degree = False, show_random = False, 
                 show_gray_overlay = False, show_original_overlay = False, 
                 show_inner_graph = False, show_mst = False, 
                 img_gr = None, original_alpha = None, 
                 ax = None, title = None, save_path = None):
    
    if ax is None:
        fig, ax = plt.subplots(figsize = (8,8))
    else:
        fig = ax.figure
    
    if original_alpha is not None:
        ax.imshow(img.img, alpha = original_alpha)
    elif show_gray_overlay:
        ax.imshow(img_gr, cmap = 'gray')
        ax.imshow(img_vor_deg, alpha = 0.25)
    elif show_original_overlay:
        ax.imshow(img.img)
        ax.imshow(img_vor_deg, alpha = 0.25)
    elif show_degree:
        ax.imshow(img_vor_deg)
    else:
        ax.imshow(img_vor)

    if show_borders:
        ax.imshow(v_border)

    if show_inner_graph or show_mst:
        xlim, ylim = ax.get_xlim(), ax.get_ylim()

        pos_inner = {n: [d['pixel_pos'][1], d['pixel_pos'][0]] for n, d in G_inner.nodes(data = True)}

        if show_inner_graph:
            nx.draw_networkx_nodes(G_inner, pos_inner, node_color = 'lightblue', node_size = 5, ax = ax, hide_ticks = False)
            nx.dray_networkx_edges(G_inner, pos_inner, edge_color = 'black', ax = ax, hide_ticks = False)

        if show_mst:
            pos_mf = {n: [d['pixel_pos'][1], d['pixel_pos'][0]] for n, d in g_msf.nodes(data = True)}
            nx.draw_networkx_edges(g_msf, pos_mf, edge_color = 'red', ax = ax, hide_ticks = False)
        
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

    if title:
        ax.set_title(title)

    if save_path is not None:
        fig.savefig(save_path, bbox_inches = 'tight', dpi = 200)

    return ax

In [ ]:
def run_power_sweep(img, label_mask, Nlabel, px_A_thm k, power = None):
    if powers is None:
        powers = np.round(np.arange(0, 1.1, 0.1), 1)

    m_values, sig_values, defect_ratio_values = [], [], []
    print(f'{'--- POWER SWEEP ---':>42}')
    print(f'{'POWER':>5} {'MEAN EDGE LENGTH':>22} {'STANDARD DEVIATION':>22} {'DEFECT RATIO':>17}')

    for p in powers:
        G = find_nodes(label_mask, Nlabel, px_a_th, img.scale)
        _, _, _, G, G_inner, _, _, = voronoi_tree(img, G, k = k, power = p)
        g_msf = nx.minimum_spanning_tree(G_inner, weight = 'dis')
        _, _, m, sig, _, defect_ratio = statistics(G, G_inner, g_msf)
        m_values.append(m)
        sig_values.append(sig)
        defect_ratio_values.append(defect_ratio)
        print(f'{p:>4.1f} {m:>18.6f} {sig:22.6f} {defect_ratio:>21.6f}')
    
    return np.asarray(powers), m_values, sig_values, defect_ratio_values

In [ ]:
def run_k_sweep(img, label_mask, Nlabel, px_a_th, power, ks = None):
    if ks is None:
        ks = np.arange(4, 25, 2)

        m_values, sig_values, defect_ratio_values = [], [], []
        print(f'{'--- K SWEEP ---':>42}')
        print(f'{'k':>5} {'MEAN EDGE LENGTH':>22} {'STANDARD DEVIATION':>22} {'DEFECT_RATIO':>17}')

        for k in ks:
            G = find_nodes(label_mask, Nlabel, px_a_th, img.scale)
            _, _, _, G, G_inner, _, _ = voronoi_tree(img, G, k = int(k), power = power)
            g_msf = nx.minimum_spanning_tree(G_inner, weight = 'dis')
            _, _, m, sig, _, defect_ratio = statistics(G, G_inner, g_msf)
            m_values.append(m)
            sig_values.append(sig)
            defect_ratio_values.append(defect_ratio)

            print(f'{k:>2.1f} {m:>18.6f} {sig:22.6f} {defect_ratio:21.6f}')
        
        return np.asarray(ks), m_values, sig_values, defect_ratio_values

In [ ]:
def plots(x, m, sig, defect_ratio, xlabel):
    fig, axes = plt.subplots(1, 3, figsize = (15, 4))
    for axis, ydata, ylab, fmt, c in ((axes[0], m, 'MEAN EDGE LENGTH', 'o-', 'red'), 
                                      (axes[1], sig, 'STANDARD DEVIAITON', 's-', 'green')
                                      (axes[2], defect_ratio, 'DEFECT RATIO', '^-', 'blue')):
        axis.plot(x, ydata, fmt, color = c, linewidth = 2, markersize = 8)
        axis.set_xlabel(xlabel, fontsize = 12)
        axis.set_ylabel(ylab, fontsize = 12)
        axis.grid(True)

    fig.tight_layout()
    return fig

In [ ]:
def lattice(img, spacing = 30, noise = 0, rng = None):
    if rng is None:
        rng = np.random.default_rng(42)

        psx, psy, scale = img.px_sizex, img.px_sizey, img.scale
        a1 = np.array([1, 0]) * spacing
        a2 = np.array([0.5, np.sqrt(3) / 2]) * spacing
        n_max = int(max(psx, psy) / spacing) + 2

        points = []
        for i in range(-n_max, n_max):
            for j in range(-n_max, n_max):
                pt = i * a1 + j * a2
                row, col = pt[1], pt[0]
                if 0 < row < psy and 0 < col < psx:
                    points.append([row, col])
        points = np.array(points, dtype = np.float32)

        if noise > 0:
            points += rng.normal(0, noise, size = points.shape)
            points[:, 0] = np.clip(points[:, 0], 0, psy - 1)
            points[:, 0] = np.clip(points[:, 1], 0, psx - 1)
        
        G = nx.Graph()
        nominal_area = (spacing ** 2) * np.sqrt(3) / 2
        for idx, pos in enumerate(points, start = 1):
            G.add_node(idx, pixel_pos = pos, area = nominal_area / scale ** 2)
        return G, points

In [ ]:
def run_validation(img, spacing = 30, noise = 0, k =8, power = 0, rng = None):
    G, points = lattice(img, spacing = spacing, noise = noise, rng = rng)
    _, _, _, G, G_inner, _, _  = voronoi_tree(img, G, k = k, power = power)
    g_msf = nx.minimum_spanning_tree(G_inner, weight = 'dis')
    _, _, m, sig, S, defect_ratio = statistics(G, G_inner, g_msf)
    return m, sig, defect_ratio, S, points

In [ ]:
def lattice_validation(img, spacing = 30, k = 8, n_noise = 15):
    rng = np.random.default_rng(42)
    m0, sig0, defect_ratio0, _, _ = run_validation(img, spacing = spacing, noise = 0, k = 8, power = 0, rng = None):
    print('--- PERFECT HEXAGONAL LATTICE ---')
    print(f'MEAN EDGE LENGTH:     {m0:.6f} nm')
    print(f'STANDARD DEVIATION:   {sig0:.2e} nm')
    print(f'DEFECT RATIO:         {defect_ratio0:.6f}')

    noise_levels = np.linspace(0, spacing * 0.4, n_noise)
    m_1, sig_1, defect_ratio_1 = [], [], []
    for ns in noise_levels:
        m, sig, defect_ratio, _, _ = run_validation(img, spacing = spacing, noise = ns, k = k, rng = rng)
        m_1.append(m)
        sig_1.append(sig)